# F7-kernels-convex-optimization — Session 1: PSD Matrices and Kernels

**85 minutes.** F6 already connected symmetric matrices, eigenvalues, and quadratic energy. Today we name the nonnegative-energy condition, prove the Gram certificate, and turn it into the definition of a valid kernel.

Keep one distinction visible throughout: **positive semidefinite** allows zero energy in nonzero directions; **positive definite** does not.


## 1. Quadratic forms: an energy audit

For a real symmetric matrix `A`, its quadratic form is

$$q_A(v)=v^T A v.$$

`A` is **positive semidefinite (PSD)** when `v^T A v >= 0` for every real vector `v` of the matching dimension. The word *semi* matters: equality may hold for a nonzero `v`. It is **positive definite (PD)** only when every nonzero `v` gives strictly positive energy.

Example: `diag(4,0)` is PSD because `4v_1^2 >= 0`, but it is not PD because `v=(0,1)` has zero energy. By contrast, `diag(4,2)` is PD.

Symmetry belongs in the definition used here. For a nonsymmetric matrix, the quadratic form sees only its symmetric part; kernel Gram matrices themselves must be symmetric.

### Checkpoint 1

1. Classify `diag(3,0,5)` as PSD, PD, both, or neither, and give the shortest certificate.
2. Find a vector with negative quadratic energy for `A=[[1,2],[2,1]]`.


## 2. The eigenvalue certificate from F6

F6 gives the spectral decomposition `A=Q Λ Q^T` for symmetric `A`. Put `c=Q^T v`. Then

$$v^T A v = c^T Λ c = \sum_i λ_i c_i^2.$$

Therefore:

- if every eigenvalue is nonnegative, every quadratic energy is nonnegative;
- if some eigenvalue `λ_j<0`, choosing its eigenvector `q_j` gives `q_j^T A q_j=λ_j<0`.

So a real symmetric matrix is PSD **if and only if** all its eigenvalues are nonnegative. Numerically, exact zeros may appear near `-1e-15`, so a declared tolerance is part of a computational audit, not a change to the mathematical definition.

### Checkpoint 2

1. Why does one negative eigenpair disprove PSD without testing any other vector?
2. If a symmetric matrix has eigenvalues `7, 7, 0`, what can you conclude about PSD, PD, and rank?


In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

A = np.array([[2.0, -2.0], [-2.0, 2.0]])
eigenvalues = np.linalg.eigvalsh(A)
is_symmetric = np.allclose(A, A.T, atol=ATOL, rtol=RTOL)
is_psd_at_tolerance = is_symmetric and eigenvalues.min() >= -ATOL
assert np.allclose(eigenvalues, np.array([0.0, 4.0]), atol=ATOL, rtol=RTOL)
assert is_psd_at_tolerance


## 3. Gram matrices are PSD — not a new miracle

F3 formed a Gram matrix from row vectors `w_1,...,w_n` by stacking them in `W` and writing `G=W W^T`. For any coefficient vector `a`,

$$a^T G a=a^T W W^T a=||W^T a||_2^2 >= 0.$$

That squared norm is a universal certificate, so every Gram matrix is PSD. It need not be PD: if the rows are linearly dependent, some nonzero `a` has `W^T a=0`.

**Worked exam-style example 1.** Let `w_1=(1,0)`, `w_2=(0,1)`, and `w_3=(1,1)`. Then

$$G=[[1,0,1],[0,1,1],[1,1,2]].$$

It is PSD by the squared-norm proof. It is not PD because `w_1+w_2-w_3=0`, so `a=(1,1,-1)` is a nonzero zero-energy direction. The repeated zero eigenvalue test is unnecessary; the dependence is already a certificate.

### Checkpoint 3

1. Give the coefficient vector that proves the Gram matrix of `u,u` is not PD when `u` is nonzero.
2. If `W` has shape `(8,3)`, what is the largest possible rank of `W W^T`, and why must it have a zero eigenvalue?


In [ ]:
W = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
G = W @ W.T
a = np.array([1.0, 1.0, -1.0])
assert np.allclose(G, np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0], [1.0, 1.0, 2.0]]), atol=ATOL, rtol=RTOL)
assert np.isclose(a @ G @ a, 0.0, atol=ATOL, rtol=RTOL)
assert np.linalg.eigvalsh(G).min() >= -ATOL


## 4. Feature maps turn inputs into Gram vectors

A **feature map** `φ` sends an input `x` to a vector `φ(x)`, possibly in a much larger space. Define

$$k(x,z)=φ(x)^T φ(z).$$

For any finite inputs `x_1,...,x_n`, stack `φ(x_i)` as rows of `Φ`. The kernel table has entries `K_ij=k(x_i,x_j)`, hence `K=Φ Φ^T`. Section 3's Gram proof immediately gives PSD.

This is the kernel trick's algebraic core: an algorithm can consume inner products in feature space without requiring us to write every feature coordinate each time.

### Checkpoint 4

1. For `φ(x)=(1,x)` on scalar inputs, write `k(x,z)` explicitly.
2. Why is symmetry automatic for any real feature-map kernel?


## 5. The finite-Gram definition of a valid kernel

A real function `k` is a **valid PSD kernel** when two separate requirements hold:

1. **symmetry:** `k(x,z)=k(z,x)` for every input pair `x,z`; and
2. **finite-Gram nonnegativity:** for every integer `n>=1`, every finite input list `x_1,...,x_n`, and every real coefficient vector `a`, the matrix entries `K_ij=k(x_i,x_j)` satisfy

$$a^TKa=\sum_{i,j} a_i a_j k(x_i,x_j) >= 0.$$

Equivalently, every finite Gram matrix is symmetric and PSD. These are two separate requirements: the quadratic form alone sees only the symmetric part of a matrix. For example, `k(x,z)=x-z` is antisymmetric. Every finite matrix it produces is skew-symmetric, so `a^TKa=0` for every real `a`, yet the function is not a valid kernel because it is not symmetric.

Both quantifiers are universal. Testing ten point clouds is not a proof about the eleventh. Conversely, one asymmetric input pair or one finite list with a negative quadratic form is enough to refute validity.

**Repeated inputs do not require strict PD.** If `x_1=x_2`, rows 1 and 2 of `K` are identical for every deterministic kernel, so `K` is singular. A valid kernel must remain PSD, not strictly PD.

### Checkpoint 5

1. Name the two separate ways to refute kernel validity with a finite certificate.
2. For repeated inputs `x,x`, name a nonzero coefficient vector that must have zero quadratic form.


## 6. Worked exam-style example 2: a degree-two feature map

For scalar inputs, consider `k(x,z)=(1+xz)^2`. Expand:

$$(1+xz)^2=1+2xz+x^2z^2.$$

Choose `φ(x)=(1, sqrt(2)x, x^2)`. Then `φ(x)^Tφ(z)=1+2xz+x^2z^2` exactly. Therefore every finite kernel matrix is `ΦΦ^T` and is PSD.

For inputs `[-1,0,2]`, constructing either the formula table or `ΦΦ^T` gives

$$K=[[4,1,1],[1,1,1],[1,1,25]].$$

The proof is the feature identity for arbitrary `x,z`. The displayed matrix is a numerical anchor, not the proof's full scope.

### Checkpoint 6

1. Why is the middle feature `sqrt(2)x` rather than `2x`?
2. If the same finite matrix has a tiny computed eigenvalue `-3e-15`, what additional information is needed before declaring a counterexample?


In [ ]:
x = np.array([-1.0, 0.0, 2.0])
Phi = np.column_stack([np.ones_like(x), np.sqrt(2.0) * x, x**2])
K_feature = Phi @ Phi.T
K_formula = (1.0 + x[:, None] * x[None, :]) ** 2
assert np.allclose(K_feature, K_formula, atol=ATOL, rtol=RTOL)
assert np.allclose(K_formula, np.array([[4.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 25.0]]), atol=ATOL, rtol=RTOL)


## 7. Common pitfalls, exam connections, and what comes next

**Pitfall — PSD means every entry is nonnegative.** Broken example: `[[1,-1],[-1,1]]` has negative entries but energies `(v_1-v_2)^2>=0`. Fix: test the quadratic form or spectrum, not entry signs.

**Pitfall — singular means invalid.** Repeated inputs force duplicate Gram rows. Fix: validity asks for PSD; zero eigenvalues are allowed.

**Pitfall — one random test proves a kernel.** A sampled spectrum examines one finite list. Fix: produce a feature map or a closure proof for validity; reserve numerical tests for evidence and counterexample search.

**Exam connection.** Round 1 questions commonly move between a kernel formula, a finite Gram matrix, a PSD classification, and a reason that sampled evidence is insufficient. Expect exact shape and quantifier language, not only an eigenvalue printout.

**Going deeper.** Session 2 develops closure proofs and constructive counterexamples. Later SVM material can consume valid kernels, but no SVM machinery is assumed here.

### Checkpoint 7

1. Give a two-by-two PSD matrix with a negative off-diagonal entry.
2. Complete the sentence precisely: a numerical PSD test on one finite point list establishes ___, while a negative eigenvalue establishes ___.


## Checkpoint answers

<details><summary><b>Checkpoint 1</b></summary>

1. PSD but not PD: all diagonal energies are nonnegative and `(0,1,0)` has zero energy. 2. `v=(1,-1)` gives `-2`.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. Its eigenvector has quadratic energy equal to that negative eigenvalue. 2. PSD, not PD, rank two.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. `(1,-1)`. 2. Rank at most three, so an eight-by-eight Gram matrix has at least five zero eigenvalues.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. `1+xz`. 2. Real dot products satisfy `φ(x)^Tφ(z)=φ(z)^Tφ(x)`.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Exhibit either one asymmetric input pair or one finite input list and coefficient vector with negative quadratic form. A passing PSD sample proves neither universal requirement. 2. `(1,-1)`.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Squaring the feature coefficient must produce the cross-term coefficient two. 2. A declared numerical tolerance and a residual/scale audit; roundoff near zero is not an exact negative witness.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. `[[1,-1],[-1,1]]`. 2. Evidence about that list; a counterexample to universal validity.

</details>
